In [48]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [84]:
import pandas as pd
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import matplotlib.pyplot as plt

In [85]:
mh_data = pd.read_csv('mental_health.csv', sep=";")
mh_data.head()

,Entity,Code,Year,schizophrenia,depressive_disorder,anxiety_disorders,bipolar_disorders,eating_disorders
0,Afghanistan,AFG,1990,0.223206,4.996118,4.713314,0.703023,0.127700
1,Afghanistan,AFG,1991,0.222454,4.989290,4.702100,0.702069,0.123256
2,Afghanistan,AFG,1992,0.221751,4.981346,4.683743,0.700792,0.118844
3,Afghanistan,AFG,1993,0.220987,4.976958,4.673549,0.700087,0.115089
4,Afghanistan,AFG,1994,0.220183,4.977782,4.670810,0.699898,0.111815


In [ ]:
mh_1990 = mh_data[mh_data['Year'] == 1990]
disorders = ["schizophrenia", "depressive_disorder", "anxiety_disorders", "bipolar_disorders", "eating_disorders"]




def calculate_disorder_percentages(df, disorders):
    """
    Calculate percentage columns for multiple disorders.

    Parameters:
    df (pd.DataFrame): The input DataFrame.
    disorders (list): A list of disorder column names to process.

    Returns:
    pd.DataFrame: The updated DataFrame with percentage columns added.
    """
    for disorder in disorders:
        percentage_column = f"{disorder}_percentage"
        df[percentage_column] = (df[disorder] / df[disorder].max()) * 100
    return df

def label_classification(percentage):
    """
    Classify a given percentage into one of three categories: 'A', 'B', or 'C'.

    Categories:
    - 'A': If the percentage is less than one-third (100 / 3) of the maximum percentage.
    - 'B': If the percentage is between one-third (100 / 3) and two-thirds (100 * 2 / 3) of the maximum percentage.
    - 'C': If the percentage is greater than or equal to two-thirds (100 * 2 / 3) of the maximum percentage.

    Parameters:
    percentage (float): The input percentage to be classified.

    Returns:
    str: A single character representing the classification ('A', 'B', or 'C').

    Examples:
    - label_classification(20) -> 'A'
    - label_classification(50) -> 'B'
    - label_classification(80) -> 'C'
    """
    if percentage < 100 / 3:
        return 'A'
    elif percentage < 100 * 2 / 3:
        return 'B'
    else:
        return 'C'


mh_1990 = calculate_disorder_percentages(mh_1990, disorders)


/var/folders/xk/yd_hy0y9731csyl_8jppq1zc0000gn/T/ipykernel_39035/2487117280.py:20: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/xk/yd_hy0y9731csyl_8jppq1zc0000gn/T/ipykernel_39035/2487117280.py:20: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/xk/yd_hy0y9731csyl_8jppq1zc0000gn/T/ipykernel_39035/2487117280.py:20: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

In [79]:




p_data['per_v'] = (p_data['cow'] / p_data['total']) * 100
p_data['per_c'] = (p_data['goat'] / p_data['total']) * 100
p_data['per_b'] = (p_data['sheep'] / p_data['total']) * 100


def label_classification(percentage):
    if percentage < 100 / 3:
        return 'A'
    elif percentage < 100 * 2 / 3:
        return 'B'
    else:
        return 'C'

p_data['v_cl'] = p_data['per_v'].apply(label_classification)
p_data['c_cl'] = p_data['per_c'].apply(label_classification)
p_data['b_cl'] = p_data['per_b'].apply(label_classification)


p_data['cheese_label'] = p_data['v_cl'] + p_data['c_cl'] + p_data['b_cl']

NameError: name 'p_data' is not defined

# Explore data

### Define lists of blended colors 

To get bivariate maps working, we need a set of nine colors that are the result of blending two main colors. Here are some examples, but others can easily be added:

1) "pink-blue" by [Joshua Stevens](http://www.joshuastevens.net/cartography/make-a-bivariate-choropleth-map/)
![Three examples of color sets](https://raw.githubusercontent.com/yotkadata/plotly-bivariate-choropleth/main/img/colors.png)

In [ ]:
# Define sets of 9 colors to be used
# Order: bottom-left, bottom-center, bottom-right, center-left, center-center, center-right, top-left, top-center, top-right
color_sets = {
    'pink-blue':   ['#e8e8e8', '#ace4e4', '#5ac8c8', '#dfb0d6', '#a5add3', '#5698b9', '#be64ac', '#8c62aa', '#3b4994'],
    'teal-red':    ['#e8e8e8', '#e4acac', '#c85a5a', '#b0d5df', '#ad9ea5', '#985356', '#64acbe', '#627f8c', '#574249'],
    'blue-organe': ['#fef1e4', '#fab186', '#f3742d',  '#97d0e7', '#b0988c', '#ab5f37', '#18aee5', '#407b8f', '#5c473d']
}

In [69]:
import pandas as pd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

# Load the dataset from CSV
df = mh_data

# Initialize the Dash app
app = dash.Dash(__name__)

# App layout
app.layout = html.Div([
    html.H4('World Mental Health Disorder Map'),
    html.P("Select a year:"),
    dcc.Dropdown(
        id='year',
        options=[{"label": year, "value": year} for year in sorted(df['Year'].unique())],
        value=df['Year'].min(),  # Default to the earliest year
        clearable=False,
    ),
    html.P("Select a disorder:"),
    dcc.Dropdown(
        id='disorder',
        options=[
            {"label": "Schizophrenia", "value": "schizophrenia"},
            {"label": "Depressive Disorder", "value": "depressive_disorder"},
            {"label": "Anxiety Disorders", "value": "anxiety_disorders"},
            {"label": "Bipolar Disorders", "value": "bipolar_disorders"},
            {"label": "Eating Disorders", "value": "eating_disorders"},
        ],
        value="schizophrenia",  # Default to "schizophrenia"
        clearable=False,
    ),
    dcc.Graph(id="graph"),
])

# Callback to update the map based on user selection
@app.callback(
    Output("graph", "figure"),
    [Input("year", "value"), Input("disorder", "value")]
)
def update_choropleth(selected_year, selected_disorder):
    # Filter the DataFrame by the selected year
    filtered_df = df[df['Year'] == selected_year]

    # Create the choropleth map
    fig = px.choropleth(
        filtered_df,
        locations="Entity",  # Matches country names in the 'Entity' column
        locationmode="country names",
        color=selected_disorder,
        hover_name="Entity",
        projection="natural earth",
        title=f"World Map of {selected_disorder.replace('_', ' ').title()} in {selected_year}"
    )
    fig.update_geos(showcoastlines=True, coastlinecolor="Black", showland=True, landcolor="lightgray")
    fig.update_layout(margin={"r": 0, "t": 30, "l": 0, "b": 0})
    return fig

# Run the app
if __name__ == "__main__":
    app.run_server(debug=True)

In [61]:
import plotly.graph_objects as go

fig = go.Figure(go.Scattergeo())
fig.update_geos(projection_type="natural earth")
fig.update_layout(height=300, margin={"r":0,"t":0,"l":0,"b":0})

In [ ]:
def conf_defaults():
    # Define some variables for later use
    conf = {
        'plot_title': 'Bivariate choropleth map using Ploty',  # Title text
        'plot_title_size': 20,  # Font size of the title
        'width': 1000,  # Width of the final map container
        'ratio': 0.8,  # Ratio of height to width
        'center_lat': 0,  # Latitude of the center of the map
        'center_lon': 0,  # Longitude of the center of the map
        'map_zoom': 3,  # Zoom factor of the map
        'hover_x_label': 'Label x variable',  # Label to appear on hover
        'hover_y_label': 'Label y variable',  # Label to appear on hover
        'borders_width': 0.5,  # Width of the geographic entity borders
        'borders_color': '#f8f8f8',  # Color of the geographic entity borders

        # Define settings for the legend
        'top': 1,  # Vertical position of the top right corner (0: bottom, 1: top)
        'right': 1,  # Horizontal position of the top right corner (0: left, 1: right)
        'box_w': 0.04,  # Width of each rectangle
        'box_h': 0.04,  # Height of each rectangle
        'line_color': '#f8f8f8',  # Color of the rectagles' borders
        'line_width': 0,  # Width of the rectagles' borders
        'legend_x_label': 'Higher x value',  # x variable label for the legend
        'legend_y_label': 'Higher y value',  # y variable label for the legend
        'legend_font_size': 9,  # Legend font size
        'legend_font_color': '#333',  # Legend font color
    }

    # Calculate height
    conf['height'] = conf['width'] * conf['ratio']

    return conf

In [ ]:
def create_legend(fig, colors, conf=conf_defaults()):

    # Reverse the order of colors
    legend_colors = colors[:]
    legend_colors.reverse()

    # Calculate coordinates for all nine rectangles
    coord = []

    # Adapt height to ratio to get squares
    width = conf['box_w']
    height = conf['box_h']/conf['ratio']

    # Start looping through rows and columns to calculate corners the squares
    for row in range(1, 4):
        for col in range(1, 4):
            coord.append({
                'x0': round(conf['right']-(col-1)*width, 4),
                'y0': round(conf['top']-(row-1)*height, 4),
                'x1': round(conf['right']-col*width, 4),
                'y1': round(conf['top']-row*height, 4)
            })

    # Create shapes (rectangles)
    for i, value in enumerate(coord):
        # Add rectangle
        fig.add_shape(go.layout.Shape(
            type='rect',
            fillcolor=legend_colors[i],
            line=dict(
                color=conf['line_color'],
                width=conf['line_width'],
            ),
            xref='paper',
            yref='paper',
            xanchor='right',
            yanchor='top',
            x0=coord[i]['x0'],
            y0=coord[i]['y0'],
            x1=coord[i]['x1'],
            y1=coord[i]['y1'],
        ))

        # Add text for first variable
        fig.add_annotation(
            xref='paper',
            yref='paper',
            xanchor='left',
            yanchor='top',
            x=coord[8]['x1'],
            y=coord[8]['y1'],
            showarrow=False,
            text=conf['legend_x_label'] + ' 🠒',
            font=dict(
                color=conf['legend_font_color'],
                size=conf['legend_font_size'],
            ),
            borderpad=0,
        )

        # Add text for second variable
        fig.add_annotation(
            xref='paper',
            yref='paper',
            xanchor='right',
            yanchor='bottom',
            x=coord[8]['x1'],
            y=coord[8]['y1'],
            showarrow=False,
            text=conf['legend_y_label'] + ' 🠒',
            font=dict(
                color=conf['legend_font_color'],
                size=conf['legend_font_size'],
            ),
            textangle=270,
            borderpad=0,
        )

    return fig

In [70]:
create_legend(fig, ['#e8e8e8', '#ace4e4', '#5ac8c8', '#dfb0d6', '#a5add3', '#5698b9', '#be64ac', '#8c62aa', '#3b4994'], )